In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OSQ_E.xpt"
csv_file = DATA_DIR / "OSQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched released source
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 OSQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OSQ010A",
    "OSQ010B",
    "OSQ010C",
    "OSQ020A",
    "OSQ020B",
    "OSQ020C",
    "OSD030AA",
    "OSQ040AA",
    "OSD050AA",
    "OSD030AB",
    "OSQ040AB",
    "OSD050AB",
    "OSD030AC",
    "OSQ040AC",
    "OSD050AC",
    "OSD030AD",
    "OSQ040AD",
    "OSD050AD",
    "OSD030AE",
    "OSQ040AE",
    "OSD050AE",
    "OSD030BA",
    "OSQ040BA",
    "OSD050BA",
    "OSD030BB",
    "OSQ040BB",
    "OSD050BB",
    "OSD030BC",
    "OSQ040BC",
    "OSD050BC",
    "OSD030BD",
    "OSQ040BD",
    "OSD030BE",
    "OSQ040BE",
    "OSD030BF",
    "OSQ040BF",
    "OSD030BG",
    "OSQ040BG",
    "OSD030CA",
    "OSQ040CA",
    "OSD050CA",
    "OSD030CB",
    "OSQ040CB",
    "OSD050CB",
    "OSD030CC",
    "OSQ040CC",
    "OSD050CC",
    "OSD030CD",
    "OSQ040CD",
    "OSD050CD",
    "OSD030CE",
    "OSQ040CE",
    "OSD050CE",
    "OSQ080",
    "OSQ090A",
    "OSQ100A",
    "OSD110A",
    "OSQ120A",
    "OSQ090B",
    "OSQ100B",
    "OSD110B",
    "OSQ120B",
    "OSQ090C",
    "OSQ100C",
    "OSD110C",
    "OSQ120C",
    "OSQ090D",
    "OSQ100D",
    "OSD110D",
    "OSQ120D",
    "OSQ090E",
    "OSQ100E",
    "OSD110E",
    "OSQ120E",
    "OSQ090F",
    "OSQ100F",
    "OSD110F",
    "OSQ120F",
    "OSQ090G",
    "OSQ100G",
    "OSD110G",
    "OSQ120G",
    "OSQ090H",
    "OSQ100H",
    "OSD110H",
    "OSQ120H",
    "OSQ090I",
    "OSQ120I",
    "OSQ090J",
    "OSQ120J",
    "OSQ060",
    "OSQ070",
    "OSQ130",
    "OSQ140Q",
    "OSQ140U",
    "OSQ150",
    "OSQ160A",
    "OSQ160B",
    "OSQ170",
    "OSQ180",
    "OSQ190",
    "OSQ200",
    "OSQ210",
    "OSQ220"
]


if df.shape != (5935, 105):
    raise ValueError(
        f"Unexpected OSQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "OSQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: OSQ_E structure = "
    "5,935 rows x 105 columns."
)


# ============================================================
# 4. CDC CHECK ITEMS
# ============================================================

#
# CDC HTML contains routing/check items:
#
# OSQ025
# OSQ045
# OSQ129
#
# These are NOT released XPT variables.
#

CHECK_ITEMS = [
    "OSQ025",
    "OSQ045",
    "OSQ129"
]


for col in CHECK_ITEMS:

    if col in df.columns:
        raise ValueError(
            f"Unexpected CHECK ITEM in XPT: {col}"
        )


print(
    "PASS: OSQ025, OSQ045 and OSQ129 "
    "correctly absent from released XPT."
)


# ============================================================
# 5. CDC EMPTY VARIABLES DROPPED FROM PUBLIC FILE
# ============================================================

#
# CDC says these completely empty fields were dropped
# from the released public file.
#
# Do NOT add them to CSV.
#

DROPPED_EMPTY_FIELDS = [
    "OSD030CF",
    "OSD030CG",
    "OSD030CH",
    "OSD030CI",
    "OSD030CJ",

    "OSD050BD",
    "OSD050BE",
    "OSD050BF",
    "OSD050BG",
    "OSD050CF",

    "OSD110I",
    "OSD110J",

    "OSQ040CF",
    "OSQ040CG",
    "OSQ040CH",
    "OSQ040CI",
    "OSQ040CJ",

    "OSQ100I",
    "OSQ100J"
]


unexpected_dropped = [
    col
    for col in DROPPED_EMPTY_FIELDS
    if col in df.columns
]


if unexpected_dropped:
    raise ValueError(
        "Unexpected empty fields present:\n"
        + str(unexpected_dropped)
    )


print(
    "PASS: CDC-documented empty variables "
    "remain absent from public XPT."
)


# ============================================================
# 6. NUMERIC FIELD CHECK
# ============================================================

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


if numeric_cols != EXPECTED_COLUMNS:
    raise ValueError(
        "Unexpected nonnumeric OSQ_E field found."
    )


print(
    "PASS: All 105 released OSQ_E fields are numeric."
)


# ============================================================
# 7. KNOWN SAS/XPORT TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY VALUE CHECK ---"
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifact found. "
        "Do NOT automatically replace with zero."
    )


print(
    "PASS: OSQ_E contains NO tiny XPORT artifacts."
)


# ============================================================
# 8. TRUE NUMERIC ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\nTrue source numeric zeros:",
    true_zero_total
)


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "Unexpected true numeric zero found."
    )


print(
    "PASS: Original OSQ_E contains no numeric zeros."
)


# ============================================================
# 9. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:
    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:
    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):
    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 10. FRACTIONAL / DECIMAL AUDIT
# ============================================================

#
# Actual attached OSQ_E:
#
# genuine fractional variables = 0
#
# Therefore every released variable can use
# integer-style formatting.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 11. CDC ANCHOR CHECK - OSQ010A
# ============================================================

EXPECTED_OSQ010A = {
    1: 72,
    2: 5858,
    9: 5
}


actual = (
    df["OSQ010A"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ010A:
    raise ValueError(
        "OSQ010A frequency mismatch."
    )


if int(
    df["OSQ010A"]
    .isna()
    .sum()
) != 0:
    raise ValueError(
        "OSQ010A missing count differs."
    )


print(
    "\nPASS: OSQ010A matches CDC."
)


# ============================================================
# 12. CDC ANCHOR CHECK - OSQ010B
# ============================================================

EXPECTED_OSQ010B = {
    1: 549,
    2: 5381,
    9: 5
}


actual = (
    df["OSQ010B"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ010B:
    raise ValueError(
        "OSQ010B frequency mismatch."
    )


if int(
    df["OSQ010B"]
    .isna()
    .sum()
) != 0:
    raise ValueError(
        "OSQ010B missing count differs."
    )


print(
    "PASS: OSQ010B matches CDC."
)


# ============================================================
# 13. CDC ANCHOR CHECK - OSQ010C
# ============================================================

EXPECTED_OSQ010C = {
    1: 153,
    2: 5775,
    9: 7
}


actual = (
    df["OSQ010C"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ010C:
    raise ValueError(
        "OSQ010C frequency mismatch."
    )


print(
    "PASS: OSQ010C matches CDC."
)


# ============================================================
# 14. CDC CHECK - OSQ020A/B/C COUNTS
# ============================================================

#
# OSQ020A:
# 1-5 valid = 72
# Missing   = 5,863
#
# OSQ020B:
# 1-7 valid = 548
# 9999      = 1
# Missing   = 5,386
#
# OSQ020C:
# 1-5 valid = 151
# 9999      = 2
# Missing   = 5,782
#

s = df["OSQ020A"]

if not (
    int(s.between(1, 5).sum()) == 72
    and
    int(s.isna().sum()) == 5863
):
    raise ValueError(
        "OSQ020A differs from CDC."
    )


s = df["OSQ020B"]

if not (
    int(s.between(1, 7).sum()) == 548
    and
    int(s.eq(9999).sum()) == 1
    and
    int(s.isna().sum()) == 5386
):
    raise ValueError(
        "OSQ020B differs from CDC."
    )


s = df["OSQ020C"]

if not (
    int(s.between(1, 5).sum()) == 151
    and
    int(s.eq(9999).sum()) == 2
    and
    int(s.isna().sum()) == 5782
):
    raise ValueError(
        "OSQ020C differs from CDC."
    )


print(
    "PASS: OSQ020A/B/C match CDC."
)


# ============================================================
# 15. CDC CHECK - OSQ060
# ============================================================

EXPECTED_OSQ060 = {
    1: 395,
    2: 5515,
    9: 25
}


actual = (
    df["OSQ060"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ060:

    raise ValueError(
        "OSQ060 frequency mismatch."
    )


if int(
    df["OSQ060"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "OSQ060 missing count differs."
    )


print(
    "\nPASS: OSQ060 matches CDC."
)


# ============================================================
# 16. CDC CHECK - OSQ070
# ============================================================

EXPECTED_OSQ070 = {
    1: 279,
    2: 114,
    9: 2
}


actual = (
    df["OSQ070"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ070:

    raise ValueError(
        "OSQ070 frequency mismatch."
    )


if int(
    df["OSQ070"]
    .isna()
    .sum()
) != 5540:

    raise ValueError(
        "OSQ070 missing count differs."
    )


print(
    "PASS: OSQ070 matches CDC."
)


# ============================================================
# 17. CDC CHECK - OSQ130
# ============================================================

EXPECTED_OSQ130 = {
    1: 278,
    2: 5619,
    9: 38
}


actual = (
    df["OSQ130"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ130:

    raise ValueError(
        "OSQ130 frequency mismatch."
    )


if int(
    df["OSQ130"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "OSQ130 missing count differs."
    )


print(
    "PASS: OSQ130 matches CDC."
)


# ============================================================
# 18. CDC CHECK - OSQ140Q
# ============================================================

#
# 1-31 = 273
# 777  = 0
# 999  = 5
# Missing = 5,657
#

s = df["OSQ140Q"]


if not (
    int(s.between(1, 31).sum()) == 273
    and
    int(s.eq(777).sum()) == 0
    and
    int(s.eq(999).sum()) == 5
    and
    int(s.isna().sum()) == 5657
):
    raise ValueError(
        "OSQ140Q differs from CDC."
    )


print(
    "PASS: OSQ140Q matches CDC."
)


# ============================================================
# 19. CDC CHECK - OSQ140U
# ============================================================

EXPECTED_OSQ140U = {
    1: 177,
    2: 96
}


actual = (
    df["OSQ140U"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ140U:

    raise ValueError(
        "OSQ140U frequency mismatch."
    )


if int(
    df["OSQ140U"]
    .isna()
    .sum()
) != 5662:

    raise ValueError(
        "OSQ140U missing count differs."
    )


print(
    "PASS: OSQ140U matches CDC."
)


# ============================================================
# 20. CDC CHECK - OSQ150
# ============================================================

EXPECTED_OSQ150 = {
    1: 594,
    2: 5054,
    7: 1,
    9: 286
}


actual = (
    df["OSQ150"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ150:

    raise ValueError(
        "OSQ150 frequency mismatch."
    )


print(
    "PASS: OSQ150 matches CDC."
)


# ============================================================
# 21. CDC CHECK - OSQ170 / OSQ180
# ============================================================

EXPECTED_OSQ170 = {
    1: 300,
    2: 5422,
    7: 1,
    9: 212
}


actual = (
    df["OSQ170"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ170:

    raise ValueError(
        "OSQ170 frequency mismatch."
    )


s = df["OSQ180"]


if not (
    int(s.between(2, 101).sum()) == 293
    and
    int(s.eq(777).sum()) == 0
    and
    int(s.eq(999).sum()) == 7
    and
    int(s.isna().sum()) == 5635
):

    raise ValueError(
        "OSQ180 differs from CDC."
    )


print(
    "PASS: OSQ170/OSQ180 match CDC."
)


# ============================================================
# 22. CDC CHECK - OSQ200 / OSQ210
# ============================================================

EXPECTED_OSQ200 = {
    1: 119,
    2: 5476,
    7: 2,
    9: 338
}


actual = (
    df["OSQ200"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_OSQ200:

    raise ValueError(
        "OSQ200 frequency mismatch."
    )


s = df["OSQ210"]


if not (
    int(s.between(14, 103).sum()) == 114
    and
    int(s.eq(777).sum()) == 0
    and
    int(s.eq(999).sum()) == 5
    and
    int(s.isna().sum()) == 5816
):

    raise ValueError(
        "OSQ210 differs from CDC."
    )


print(
    "PASS: OSQ200/OSQ210 match CDC."
)


# ============================================================
# 23. EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# OSQ_E requires NO numeric corrections.
#
# Tiny artifacts = 0
# Zero restorations = 0
# Genuine decimals = 0
#
# Therefore expected_df is EXACTLY the original XPT.
#

expected_df = source_df.copy()


# ============================================================
# 24. VERIFY ALL VALUES ARE WHOLE BEFORE INT64
# ============================================================

for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ):

        raise ValueError(
            f"STOP: {col} contains a non-whole value."
        )


print(
    "\nPASS: All 105 released OSQ_E fields "
    "contain whole-number source values."
)


# ============================================================
# 25. CLEAN WHOLE-NUMBER FORMATTING
# ============================================================

#
# Every released OSQ_E variable has been proven whole.
#
# Examples:
#
# SEQN
# 41475.0 -> 41475
#
# OSQ010A
# 1.0 -> 1
#
# OSD030AA
# 80.0 -> 80
#
# OSQ140Q
# 999.0 -> 999
#
# Missing
# NaN -> blank
#

for col in EXPECTED_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All OSQ_E variables converted "
    "to clean integer-style formatting."
)


# ============================================================
# 26. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected pre-export changes:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal "
    "original XPT exactly."
)


# ============================================================
# 27. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO:
#
# round()
# float_format
# arbitrary recoding
# missing-value filling
#
# Missing source values become blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 28. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 29. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 30. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 105 columns numerically "
    "match original XPT exactly."
)


# ============================================================
# 31. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-position fidelity failure:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 32. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 5935:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 33. STRICT RAW CSV TOKEN FIDELITY
# ============================================================

#
# Since all OSQ_E variables are whole-number fields:
#
# expected raw token =
# str(int(source_value))
#
# Missing -> blank
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )

    for example in raw_examples:
        print(example)


    raise ValueError(
        "OSQ_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected XPT-derived representation."
)


# ============================================================
# 34. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in EXPECTED_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Unwanted '.0' values:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 35. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 36. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 37. FINAL ZERO CHECK
# ============================================================

final_zero_total = int(
    csv_df[
        EXPECTED_COLUMNS
    ]
    .eq(0)
    .sum()
    .sum()
)


if final_zero_total != 0:

    raise ValueError(
        "Unexpected zero value introduced."
    )


print(
    "PASS: No unexpected zero values introduced."
)


# ============================================================
# 38. FINAL CDC ANCHOR RECHECK
# ============================================================

if (
    csv_df["OSQ010A"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
    !=
    {
        1: 72,
        2: 5858,
        9: 5
    }
):

    raise ValueError(
        "Final OSQ010A distribution changed."
    )


if (
    csv_df["OSQ060"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
    !=
    {
        1: 395,
        2: 5515,
        9: 25
    }
):

    raise ValueError(
        "Final OSQ060 distribution changed."
    )


s = csv_df["OSQ140Q"]

if not (
    int(
        s.between(
            1,
            31
        ).sum()
    ) == 273
    and
    int(s.eq(999).sum()) == 5
    and
    int(s.isna().sum()) == 5657
):

    raise ValueError(
        "Final OSQ140Q distribution changed."
    )


s = csv_df["OSQ180"]

if not (
    int(
        s.between(
            2,
            101
        ).sum()
    ) == 293
    and
    int(s.eq(999).sum()) == 7
    and
    int(s.isna().sum()) == 5635
):

    raise ValueError(
        "Final OSQ180 distribution changed."
    )


s = csv_df["OSQ210"]

if not (
    int(
        s.between(
            14,
            103
        ).sum()
    ) == 114
    and
    int(s.eq(999).sum()) == 5
    and
    int(s.isna().sum()) == 5816
):

    raise ValueError(
        "Final OSQ210 distribution changed."
    )


print(
    "PASS: Final CDC/XPT anchor distributions unchanged."
)


# ============================================================
# 39. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("OSQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nTiny artifacts:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    true_zero_total
)

print(
    "Genuine decimal variables:",
    0
)


print(
    "\nCDC CHECK ITEMS not added:"
)

for item in CHECK_ITEMS:
    print(item)


print(
    "\nCDC empty variables remain dropped:"
)

for item in DROPPED_EMPTY_FIELDS:
    print(item)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "OSQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 OSQ_E XPT -> CSV
Rows: 5,935
Columns: 105
PASS: OSQ_E structure = 5,935 rows x 105 columns.
PASS: OSQ025, OSQ045 and OSQ129 correctly absent from released XPT.
PASS: CDC-documented empty variables remain absent from public XPT.
PASS: All 105 released OSQ_E fields are numeric.

--- ORIGINAL XPT TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: OSQ_E contains NO tiny XPORT artifacts.

True source numeric zeros: 0
PASS: Original OSQ_E contains no numeric zeros.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
Genuine fractional variables: {}
PASS: Genuine decimal variables = 0.

PASS: OSQ010A matches CDC.
PASS: OSQ010B matches CDC.
PASS: OSQ010C matches CDC.
PASS: OSQ020A/B/C match CDC.

PASS: OSQ060 matches CDC.
PASS: OSQ070 matches CDC.
PASS: OSQ130 matches CDC.
PASS: OSQ140Q matches CDC.
PASS: OSQ140U matches CDC.
PASS: OSQ150 matches CDC.
PASS: OSQ170/OSQ180 match CDC.
PASS: OSQ200/OSQ210 match CD